# Causal Impact of Delivery Delay on Customer Review Scores & Retention (Sharp RDD)

**Research Question:** Does crossing the promised delivery deadline causally reduce customer review scores and future repeat purchase probability in e-commerce?

**Dataset:** Olist Brazilian E-Commerce Dataset (9 relational tables; 96,470 delivered orders).

**Empirical Strategy:**
- **Running Variable:** `delay_days = floor(delivered_date) - floor(estimated_delivery_date)`
- **Sharp Cutoff:** `is_delayed = 1(delay_days > 0)`
- **Main Specification:** Local linear regression within symmetric bandwidth `[-7, 7]` with Seller, Product Category, and Year-Month Fixed Effects (`fixest::feols`), standard errors clustered by `main_seller_id`.
- **Robustness Checks:** Bandwidth sensitivity (`[-5, 5]`, `[-7, 7]`, `[-10, 10]`), pre-determined covariate controls (`total_price`, `total_freight`, `product_weight_g`), Triangular Kernel weighting, and discrete-adjusted `rdrobust` comparison.

## 1. Data Loading & Relational Schema Audit (Python)

Load all 9 Olist tables from Kaggle and check key uniqueness (`Strict_1to1`) before merging to prevent `1-to-Many` join explosions.

In [ ]:
import os
import kagglehub
import numpy as np
import pandas as pd

# Download Olist dataset via kagglehub
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
print("Dataset path:", path)

files = {
    "orders": "olist_orders_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "cat_translation": "product_category_name_translation.csv",
}

dfs = {name: pd.read_csv(os.path.join(path, fname)) for name, fname in files.items()}
print(f"Loaded {len(dfs)} relational tables successfully.")

In [ ]:
# Audit primary/foreign key cardinality across tables
key_checks = [
    ("orders", "order_id"),
    ("customers", "customer_id"),
    ("customers", "customer_unique_id"),
    ("items", "order_id"),
    ("payments", "order_id"),
    ("reviews", "order_id"),
    ("reviews", "review_id"),
    ("sellers", "seller_id"),
    ("products", "product_id"),
    ("geolocation", "geolocation_zip_code_prefix"),
]

summary = []
for table_name, key in key_checks:
    df_tmp = dfs[table_name]
    n_rows = len(df_tmp)
    n_unique = df_tmp[key].nunique(dropna=True)
    summary.append({
        "Table": table_name,
        "Key": key,
        "Total_Rows": n_rows,
        "Unique_Keys": n_unique,
        "Strict_1to1": (n_rows == n_unique),
    })

display(pd.DataFrame(summary))

print("\n--- Orders Status Distribution ---")
print(dfs["orders"]["order_status"].value_counts(dropna=False))

print("\n--- Orders Missing Timestamps ---")
print(dfs["orders"].isnull().sum())

## 2. 1-to-1 Pre-Aggregation & Master Table Construction

Because `payments`, `items`, and `reviews` are `1-to-Many` at the `order_id` level, we pre-aggregate each table to strict `1-to-1` granularity before left-joining onto delivered orders (`N = 96,470`), and verify row-count preservation with an explicit `assert`.

In [ ]:
# 1. Base table: delivered orders with valid delivery and estimated timestamps
orders_clean = dfs["orders"][
    (dfs["orders"]["order_status"] == "delivered")
    & (dfs["orders"]["order_delivered_customer_date"].notnull())
    & (dfs["orders"]["order_estimated_delivery_date"].notnull())
].copy()

print(f"Valid delivered orders: {len(orders_clean):,}")

# 2. Pre-aggregate payments at order_id level
payments_agg = dfs["payments"].groupby("order_id", as_index=False).agg(
    total_payment=("payment_value", "sum"),
    max_installments=("payment_installments", "max"),
    n_payment_methods=("payment_sequential", "count"),
)

# 3. Pre-aggregate items at order_id level
# Retain highest-priced item's seller_id and product_id as primary FE attributes
items_sorted = dfs["items"].sort_values(["order_id", "price"], ascending=[True, False])
items_agg = items_sorted.groupby("order_id", as_index=False).agg(
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum"),
    n_items=("order_item_id", "count"),
    n_sellers=("seller_id", "nunique"),
    main_seller_id=("seller_id", "first"),
    main_product_id=("product_id", "first"),
)

# 4. Deduplicate reviews at order_id level (keep most recent review)
reviews_clean = (
    dfs["reviews"]
    .sort_values("review_creation_date", ascending=False)
    .drop_duplicates(subset=["order_id"], keep="first")
    [["order_id", "review_score", "review_creation_date"]]
)

# 5. Merge into 1-to-1 master order table
df_master = (
    orders_clean
    .merge(
        dfs["customers"][["customer_id", "customer_unique_id", "customer_zip_code_prefix", "customer_state"]],
        on="customer_id",
        how="left",
    )
    .merge(payments_agg, on="order_id", how="left")
    .merge(items_agg, on="order_id", how="left")
    .merge(reviews_clean, on="order_id", how="left")
    .merge(
        dfs["products"][["product_id", "product_category_name", "product_weight_g"]],
        left_on="main_product_id",
        right_on="product_id",
        how="left",
    )
    .merge(dfs["cat_translation"], on="product_category_name", how="left")
)

# Programmatic integrity check: prevent join explosions
assert len(df_master) == len(orders_clean), "Error: Row count expanded during merge!"
print(f"Master table verified 1-to-1: {len(df_master):,} rows x {df_master.shape[1]} columns")
display(df_master.head(5))

## 3. Feature Engineering & Right-Censoring Adjustment

- **Date-level truncation (`.dt.floor("D")`):** Ensures orders delivered on the afternoon of the estimated date are classified as on-time (`delay_days == 0`), placing the Sharp RDD cutoff strictly at `delay_days > 0`.
- **Repeat Purchase & Right-Censoring:** Uses `customer_unique_id` sorted chronologically to track future orders, flags first-time orders (`is_first_order == 1`) to avoid within-customer serial correlation, and flags orders with a full 90-day post-purchase observation window (`valid_90d_window == 1`).

In [ ]:
# 1. Construct running variable (delay_days) and treatment indicator (is_delayed)
delivered_date = pd.to_datetime(df_master["order_delivered_customer_date"]).dt.floor("D")
estimated_date = pd.to_datetime(df_master["order_estimated_delivery_date"]).dt.floor("D")

df_master["delay_days"] = (delivered_date - estimated_date).dt.days
df_master["is_delayed"] = (df_master["delay_days"] > 0).astype(int)

# 2. Construct Year-Month fixed effect attribute
df_master["order_purchase_dt"] = pd.to_datetime(df_master["order_purchase_timestamp"])
df_master["purchase_ym"] = df_master["order_purchase_dt"].dt.to_period("M").astype(str)

# 3. Track repeat purchase behavior via customer_unique_id
df_master = df_master.sort_values(["customer_unique_id", "order_purchase_dt"]).reset_index(drop=True)
df_master["next_purchase_dt"] = df_master.groupby("customer_unique_id")["order_purchase_dt"].shift(-1)
df_master["days_to_next_order"] = (
    (df_master["next_purchase_dt"] - df_master["order_purchase_dt"]).dt.total_seconds() / (24 * 3600)
)

df_master["has_repeat_order"] = df_master["next_purchase_dt"].notnull().astype(int)
df_master["repeat_90d"] = (
    df_master["days_to_next_order"].notnull() & (df_master["days_to_next_order"] <= 90)
).astype(int)

# First-order flag & 90-day right-censoring adjustment
df_master["order_seq"] = df_master.groupby("customer_unique_id").cumcount() + 1
df_master["is_first_order"] = (df_master["order_seq"] == 1).astype(int)

max_date = df_master["order_purchase_dt"].max()
df_master["valid_90d_window"] = (
    df_master["order_purchase_dt"] <= (max_date - pd.Timedelta(days=90))
).astype(int)

# 4. Descriptive sanity checks
print("--- Core Summary Statistics ---")
print(f"Total delivered orders:   {len(df_master):,}")
print(f"Late delivery rate (>0d): {df_master['is_delayed'].mean():.2%}")
print(f"Overall mean review:      {df_master['review_score'].mean():.3f}")

print("\nNaive Review Score Comparison (On-Time vs. Delayed):")
print(df_master.groupby("is_delayed")["review_score"].agg(["count", "mean", "std"]))

print("\nFirst-Time Buyer Retention (Valid 90-Day Window):")
first_valid = df_master[(df_master["is_first_order"] == 1) & (df_master["valid_90d_window"] == 1)]
print(first_valid.groupby("is_delayed")[["has_repeat_order", "repeat_90d"]].mean())

# 5. Export clean dataset for R econometric estimation
df_master.to_csv("olist_rdd_clean.csv", index=False)
print("\nExported clean dataset to olist_rdd_clean.csv")

## 4. R Environment Setup & RDD Validity Checks (`rpy2`)

Switch to R to run high-dimensional fixed-effects regressions (`fixest`), density checks, and covariate balance placebo tests around the cutoff `[-7, 7]`.

In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R
required_pkgs <- c("fixest", "rdrobust", "dplyr", "ggplot2")
for (pkg in required_pkgs) {
  if (!require(pkg, character.only = TRUE)) {
    install.packages(pkg, repos = "https://cloud.r-project.org/")
    library(pkg, character.only = TRUE)
  }
}

df <- read.csv("olist_rdd_clean.csv")
df_first <- df %>% filter(is_first_order == 1)
cat("Loaded R dataset:", nrow(df), "total rows |", nrow(df_first), "first-time orders\n")

In [ ]:
%%R
# 1. Order density around the cutoff [-7, 7] (check for manipulation / bunching)
df_bw7 <- df %>% filter(delay_days >= -7, delay_days <= 7)
cat("--- Order Count by delay_days in [-7, 7] ---\n")
print(table(df_bw7$delay_days))

# 2. Covariate Balance / Placebo RDD Tests on pre-determined order attributes
bal_price <- feols(
  total_price ~ is_delayed * delay_days | main_seller_id + product_category_name_english + purchase_ym,
  data = df_bw7, vcov = ~main_seller_id
)
bal_freight <- feols(
  total_freight ~ is_delayed * delay_days | main_seller_id + product_category_name_english + purchase_ym,
  data = df_bw7, vcov = ~main_seller_id
)
bal_weight <- feols(
  product_weight_g ~ is_delayed * delay_days | main_seller_id + product_category_name_english + purchase_ym,
  data = df_bw7, vcov = ~main_seller_id
)

print(etable(
  bal_price, bal_freight, bal_weight,
  keep = "is_delayed",
  headers = c("Order Price", "Freight Value", "Product Weight"),
  title = "RDD Covariate Balance Test (Bandwidth = [-7, 7])"
))

## 5. Main Sharp RDD Estimation (`fixest`, Bandwidth = `[-7, 7]`)

Estimate local linear RDD models on first-time buyers (`is_first_order == 1`) with seller-clustered standard errors:
1. **Model (1):** Review Score without Fixed Effects (baseline)
2. **Model (2):** Review Score with Seller + Category + Year-Month Fixed Effects (preferred specification)
3. **Model (3):** Future Repeat Order (Any) with Full Fixed Effects
4. **Model (4):** 90-Day Repeat Order (Right-Censoring Adjusted) with Full Fixed Effects

In [ ]:
%%R
df_first_bw7 <- df_first %>% filter(delay_days >= -7, delay_days <= 7)
df_first_90d_bw7 <- df_first_bw7 %>% filter(valid_90d_window == 1)

m1_score <- feols(
  review_score ~ is_delayed * delay_days,
  data = df_first_bw7, vcov = ~main_seller_id
)

m2_score <- feols(
  review_score ~ is_delayed * delay_days | main_seller_id + product_category_name_english + purchase_ym,
  data = df_first_bw7, vcov = ~main_seller_id
)

m3_rep_any <- feols(
  has_repeat_order ~ is_delayed * delay_days | main_seller_id + product_category_name_english + purchase_ym,
  data = df_first_bw7, vcov = ~main_seller_id
)

m4_rep_90d <- feols(
  repeat_90d ~ is_delayed * delay_days | main_seller_id + product_category_name_english + purchase_ym,
  data = df_first_90d_bw7, vcov = ~main_seller_id
)

print(etable(
  m1_score, m2_score, m3_rep_any, m4_rep_90d,
  keep = c("is_delayed", "delay_days"),
  headers = c("Score (No FE)", "Score (Full FE)", "Repeat (Any)", "Repeat (90d)"),
  fitstat = ~ n + r2 + ar2
))

# Export publication-ready LaTeX table
etable(
  m1_score, m2_score, m3_rep_any, m4_rep_90d,
  keep = c("is_delayed", "delay_days"),
  headers = c("Score (No FE)", "Score (Full FE)", "Repeat (Any)", "Repeat (90d)"),
  tex = TRUE,
  file = "rdd_main_results.tex",
  replace = TRUE
)
cat("\nExported main regression table to rdd_main_results.tex\n")

## 6. Robustness Checks: Bandwidth Sensitivity, Controls, Triangular Kernel & Discrete `rdrobust`

1. **Bandwidth Sensitivity & Covariate Controls:** Test `[-5, 5]`, `[-7, 7]` (main), `[-10, 10]`, and `[-7, 7]` augmented with `total_price + total_freight + product_weight_g` (addressing the slight post-cutoff price slope noted in Section 4).
2. **Triangular Kernel Weighting:** Assign higher weights to observations closer to Day 0 while retaining full Fixed Effects and seller-clustered SEs.
3. **Discrete-Adjusted `rdrobust` Comparison:** Because `delay_days` is an integer-valued discrete running variable (`mass points`), automatic narrow bandwidths (`h ~ 3.8`) inflate quadratic bias-correction variance. Aligning `h = 7` and reporting transparently highlights that the **post-cutoff slope steepening (`is_delayed × delay_days`)** is the most robust causal signal.

In [ ]:
%%R
# 1. Bandwidth sensitivity & covariate controls
m_bw5 <- feols(
  review_score ~ is_delayed * delay_days | main_seller_id + product_category_name_english + purchase_ym,
  data = df_first %>% filter(delay_days >= -5, delay_days <= 5),
  vcov = ~main_seller_id
)

m_bw10 <- feols(
  review_score ~ is_delayed * delay_days | main_seller_id + product_category_name_english + purchase_ym,
  data = df_first %>% filter(delay_days >= -10, delay_days <= 10),
  vcov = ~main_seller_id
)

m_bw7_ctrl <- feols(
  review_score ~ is_delayed * delay_days + total_price + total_freight + product_weight_g |
    main_seller_id + product_category_name_english + purchase_ym,
  data = df_first_bw7,
  vcov = ~main_seller_id
)

# 2. Triangular Kernel weighted local linear RDD on [-7, 7]
df_bw7_tri <- df_first_bw7 %>%
  mutate(tri_weight = 1 - abs(delay_days) / 8)

m_bw7_tri <- feols(
  review_score ~ is_delayed * delay_days | main_seller_id + product_category_name_english + purchase_ym,
  data = df_bw7_tri,
  weights = ~tri_weight,
  vcov = ~main_seller_id
)

print(etable(
  m_bw5, m2_score, m_bw10, m_bw7_ctrl, m_bw7_tri,
  keep = c("is_delayed", "delay_days"),
  headers = c("BW [-5,5]", "BW [-7,7] (Main)", "BW [-10,10]", "BW [-7,7] + Ctrls", "Triangular Kernel"),
  fitstat = ~ n + r2 + ar2
))

etable(
  m_bw5, m2_score, m_bw10, m_bw7_ctrl, m_bw7_tri,
  keep = c("is_delayed", "delay_days"),
  headers = c("BW [-5,5]", "BW [-7,7] (Main)", "BW [-10,10]", "BW [-7,7] + Ctrls", "Triangular Kernel"),
  tex = TRUE,
  file = "rdd_robustness_results.tex",
  replace = TRUE
)
cat("\nExported robustness table to rdd_robustness_results.tex\n")

In [ ]:
%%R
# 3. Non-parametric rdrobust comparison (discrete running variable with masspoints adjustment)
cat("--- rdrobust (Aligned h = 7, Discrete Mass Points Adjusted) ---\n")
rdr_h7 <- rdrobust(
  y = df_first$review_score,
  x = df_first$delay_days,
  c = 0.5,
  h = 7,
  masspoints = "adjust"
)
summary(rdr_h7)

## 7. Visual Evidence (`ggplot2`)

Plot daily mean review scores with 95% confidence intervals and separate linear fits on either side of the delivery cutoff across `[-10, 10]` days for broader visual context.

In [ ]:
%%R
plot_df <- df_first %>%
  filter(delay_days >= -10, delay_days <= 10, !is.na(review_score)) %>%
  group_by(delay_days, is_delayed) %>%
  summarise(
    mean_score = mean(review_score),
    se = sd(review_score) / sqrt(n()),
    .groups = "drop"
  )

p <- ggplot(plot_df, aes(x = delay_days, y = mean_score, color = factor(is_delayed))) +
  geom_point(size = 2.5) +
  geom_errorbar(aes(ymin = mean_score - 1.96 * se, ymax = mean_score + 1.96 * se), width = 0.25) +
  geom_smooth(method = "lm", formula = y ~ x, se = TRUE) +
  geom_vline(xintercept = 0.5, linetype = "dashed", color = "black") +
  scale_color_manual(
    values = c("#2b5c8f", "#d95f02"),
    labels = c("On-Time (<= 0)", "Delayed (> 0)")
  ) +
  scale_x_continuous(breaks = seq(-10, 10, by = 2)) +
  ylim(1, 5) +
  labs(
    title = "Sharp RDD: Causal Effect of Delivery Delay on Review Score",
    subtitle = "First-time buyers within [-10, 10] days around estimated delivery date",
    x = "Delivery Delay (Actual - Estimated Days)",
    y = "Average Review Score (1-5)",
    color = NULL
  ) +
  theme_minimal(base_size = 12) +
  theme(legend.position = "top")

ggsave("rdd_review_score_plot.png", plot = p, width = 8, height = 5, dpi = 300)
print(p)